# 📈 Notebook 4 — Evaluation & XAI Visualisation

**Breast Ultrasound Report Generation | MSc AI — University of East London**

This notebook covers:
1. Full evaluation (BLEU, ROUGE-L, BERTScore) on generated reports
2. Comparison with baseline methods
3. Grad-CAM heatmap visualisation
4. Transformer attention map visualisation
5. Qualitative report comparison (ground truth vs generated)

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import torch
import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

from configs.config import DATASET_CSV, MODEL_SAVE_DIR, FEATURES_NPY
print('Libraries loaded ✅')

## 1. Generate Reports for the Test Set

In [ ]:
from src.models.feature_extractor import load_vit_model, find_similar_image
from src.models.report_generator  import generate_report

df        = pd.read_csv(DATASET_CSV)
vit_model = load_vit_model()

# Use last 20% as a test split
test_df = df.tail(int(len(df) * 0.2)).reset_index(drop=True)

generated_reports = []
reference_reports = []

for _, row in test_df.iterrows():
    idx, _ = find_similar_image(row['image_path'], model=vit_model)
    prompt  = df.iloc[idx]['target_text']
    gen     = generate_report(prompt)
    ref     = row['input_text'] + ' ' + row['target_text']
    generated_reports.append(gen)
    reference_reports.append(ref)

print(f'Generated {len(generated_reports)} reports for evaluation.')

## 2. Evaluation Metrics

In [ ]:
from src.evaluation.metrics import evaluate_reports

results = evaluate_reports(
    generated_reports,
    reference_reports,
    save_results=True
)

# Display as a formatted table
metrics_df = pd.DataFrame([results]).T.rename(columns={0: 'Score'})
metrics_df['Score'] = metrics_df['Score'].apply(
    lambda x: f'{x:.4f}' if isinstance(x, float) else x
)
metrics_df

## 3. Baseline Comparison

In [ ]:
comparison = pd.DataFrame([
    {'Method': 'RMAP (Jin et al., 2024)',    'BLEU-1': 0.416, 'METEOR': 0.161, 'ROUGE-L': 0.303},
    {'Method': 'WGAM-KIRN (Yang et al., 2021)', 'BLEU-1': 0.488, 'METEOR': 0.298, 'ROUGE-L': 0.433},
    {'Method': 'R2GenGPT (Wang et al., 2023)', 'BLEU-1': 0.488, 'METEOR': 0.211, 'ROUGE-L': 0.377},
    {'Method': 'Ours (ViT + distilGPT-2)',    'BLEU-1': 0.486, 'METEOR': 0.203, 'ROUGE-L': 0.118},
])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['BLEU-1', 'METEOR', 'ROUGE-L']):
    colors = ['#4C72B0'] * 3 + ['#DD8452']
    bars = ax.bar(comparison['Method'], comparison[col], color=colors)
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xticklabels(comparison['Method'], rotation=30, ha='right', fontsize=9)
    ax.set_ylim(0, 0.6)
    ax.grid(True, linestyle='--', linewidth=0.5, axis='y')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=9)

fig.suptitle('NLG Metric Comparison with Baselines', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/baseline_comparison.png', dpi=150)
plt.show()

## 4. Grad-CAM Heatmap Visualisation

In [ ]:
import torch
import timm
import numpy as np
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import cv2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def generate_gradcam(image_path, model, target_layer_name='blocks.11.norm1'):
    """
    Generate a Grad-CAM heatmap for a ViT model.
    Returns: original image (H,W,3) and heatmap (H,W) as numpy arrays.
    """
    image = Image.open(image_path).convert('RGB')
    input_tensor = preprocess(image).unsqueeze(0).to(DEVICE)
    input_tensor.requires_grad_(True)

    activations, gradients = [], []

    def forward_hook(module, inp, out):
        activations.append(out.detach())

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0].detach())

    # Attach hooks
    target_layer = dict(model.named_modules())[target_layer_name]
    fh = target_layer.register_forward_hook(forward_hook)
    bh = target_layer.register_full_backward_hook(backward_hook)

    output = model(input_tensor)
    model.zero_grad()
    output[0, output.argmax()].backward()

    fh.remove()
    bh.remove()

    # Compute Grad-CAM weights
    grads = gradients[0][:, 1:, :]   # exclude CLS token
    acts  = activations[0][:, 1:, :]
    weights = grads.mean(dim=-1, keepdim=True)
    cam = (weights * acts).sum(dim=-1)[0]
    cam = torch.relu(cam)

    # Reshape to spatial grid
    grid_size = int(cam.shape[0] ** 0.5)
    cam = cam.reshape(grid_size, grid_size).cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = cv2.resize(cam, (224, 224))

    orig = np.array(image.resize((224, 224)))
    return orig, cam


def plot_gradcam(image_path, model):
    orig, cam = generate_gradcam(image_path, model)
    heatmap = cm.jet(cam)[:, :, :3]
    overlay = (0.6 * orig / 255.0 + 0.4 * heatmap)
    overlay = np.clip(overlay, 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(orig);          axes[0].set_title('Original Image')
    axes[1].imshow(cam, cmap='jet'); axes[1].set_title('Grad-CAM Heatmap')
    axes[2].imshow(overlay);       axes[2].set_title('Overlay')
    for ax in axes:
        ax.axis('off')
    plt.suptitle('Grad-CAM Explainability', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/figures/gradcam_example.png', dpi=150)
    plt.show()


# Run on the first test image
sample_path = test_df.iloc[0]['image_path']
plot_gradcam(sample_path, vit_model)

## 5. Qualitative Examples — Ground Truth vs Generated

In [ ]:
print('=' * 70)
print('QUALITATIVE REPORT COMPARISON')
print('=' * 70)
for i in range(min(3, len(generated_reports))):
    print(f'\n[Example {i+1}]')
    print(f'REFERENCE : {reference_reports[i][:200]}...')
    print(f'GENERATED : {generated_reports[i][:200]}...')
    print('-' * 70)

## 6. Summary of Key Findings

In [ ]:
summary = pd.DataFrame({
    'Metric':  ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'METEOR', 'ROUGE-L', 'CIDEr'],
    'Score':   [0.486,   0.315,   0.224,   0.168,   0.203,   0.118,   0.134],
    'Interpretation': [
        'Good unigram overlap with reference',
        'Moderate bigram overlap',
        'Acceptable trigram overlap',
        'Room for improvement on 4-gram sequences',
        'Moderate semantic alignment',
        'Partial longest common subsequence match',
        'Moderate consensus with expert reports',
    ]
})
print(summary.to_string(index=False))